In [3]:
# ==============================================================================
# Project Sentinel: A CAI-Powered UEBA Platform for Fraud Detection
# Author: Aishwary Prakash Singh
# ==============================================================================

# --- Imports ---
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import random
from typing import Dict, Any

# ==============================================================================
# 1. UTILITY FUNCTIONS FOR OUTPUT FORMATTING
# ==============================================================================

def print_header(title: str):
    """Prints a formatted main header for a section."""
    print("\n" + "="*70)
    print(f"    {title.upper()}")
    print("="*70)

def print_subheader(title: str):
    """Prints a formatted subheader."""
    print(f"\n--- {title} ---")

def print_result(label: str, value: Any, indent: int = 1):
    """Prints a formatted key-value pair for displaying results."""
    prefix = "  " * indent
    print(f"{prefix}{label:<28}: {value}")

# ==============================================================================
# 2. MOCK DATA AND MODEL GENERATION
# ==============================================================================

def get_mock_historical_data() -> pd.DataFrame:
    """
    Creates a realistic mock dataset where fraudulent transactions have
    distinct, learnable patterns compared to normal transactions.
    """
    # Normal transactions (95% of data)
    normal_data = {
        'amount': np.random.normal(800, 400, 950),
        'hour_of_day': np.random.randint(7, 22, 950), # Mostly daytime hours
        'is_new_beneficiary': np.random.randint(0, 2, 950),
        'device_changes_last_24h': np.random.randint(0, 2, 950),
        'is_fraud': [0] * 950
    }

    # Fraudulent transactions (5% of data)
    fraud_data = {
        'amount': np.random.normal(8000, 1000, 50), # Significantly higher amounts
        'hour_of_day': np.random.randint(0, 6, 50), # Mostly late night/early morning
        'is_new_beneficiary': [1] * 50, # Always a new beneficiary for this pattern
        'device_changes_last_24h': np.random.randint(2, 5, 50), # High number of device changes
        'is_fraud': [1] * 50
    }

    df = pd.concat([pd.DataFrame(normal_data), pd.DataFrame(fraud_data)])
    return df.sample(frac=1, random_state=42).reset_index(drop=True) # Shuffle data

def get_mock_mule_network() -> Dict[str, list]:
    """Creates a mock dictionary representing known mule account networks."""
    return {
        "CLUSTER_01_LAUNDERING_RING": ["account_C", "account_D", "account_E"],
        "CLUSTER_02_PHISHING_GROUP": ["account_X", "account_Y", "account_Z_HUB"]
    }

# ==============================================================================
# 3. THE CORE FRAUD DETECTION ENGINE
# ==============================================================================

class FraudDetectionEngine:
    """
    Simulates the core logic of Project Sentinel, featuring a multi-layered
    AI approach for real-time fraud detection.
    """
    def __init__(self):
        """Initializes and trains all the necessary machine learning models."""
        print_header("Initializing Project Sentinel Engine")
        self.historical_data = get_mock_historical_data()
        self.mule_network = get_mock_mule_network()
        self.scaler = StandardScaler()

        # Model 1: Anomaly Detection (Unsupervised)
        self.anomaly_detector = IsolationForest(contamination=0.1, random_state=42)
        anomaly_features = self.scaler.fit_transform(self.historical_data[['amount', 'hour_of_day']])
        self.anomaly_detector.fit(anomaly_features)
        print_result("Status", "Anomaly Detection model trained.", 0)

        # Model 2: Fraud Prediction (Supervised)
        self.fraud_predictor = xgb.XGBClassifier(eval_metric='logloss', random_state=42)
        X = self.historical_data.drop('is_fraud', axis=1)
        y = self.historical_data['is_fraud']
        self.fraud_predictor.fit(X, y)
        print_result("Status", "Fraud Prediction (XGBoost) model trained.", 0)

        print("\nEngine is ready for real-time analysis.")

    def _get_behavioral_features(self, transaction: Dict[str, Any]) -> Dict[str, int]:
        """Simulates fetching historical user data to provide behavioral context."""
        # Use specific features for demo transactions to ensure clear results
        if transaction['transaction_id'] == 'txn_456':
             return {'device_changes_last_24h': 3}
        return {'device_changes_last_24h': random.randint(0, 1)}

    def _analyze_graph_network(self, transaction: Dict[str, Any]) -> int:
        """Simulates checking if an account is part of a known mule network."""
        sender, receiver = transaction['sender_account'], transaction['receiver_account']
        for cluster_id, accounts in self.mule_network.items():
            if sender in accounts or receiver in accounts:
                print_result("Graph Analysis Alert", f"Account linked to {cluster_id}!")
                return 30 # Return a significant risk penalty
        print_result("Graph Analysis Alert", "No links to known fraud networks found.")
        return 0

    def predict(self, transaction: Dict[str, Any]) -> Dict[str, Any]:
        """
        Processes a single transaction through the full analysis pipeline
        and generates a structured report of the findings.
        """
        print_header(f"Analysis Report for Transaction ID: {transaction['transaction_id']}")

        # --- Stage 1: Feature Engineering & Context ---
        print_subheader("Stage 1: Feature Engineering")
        behavioral_features = self._get_behavioral_features(transaction)
        features_for_prediction = pd.DataFrame([{
            'amount': transaction['amount'],
            'hour_of_day': transaction['hour_of_day'],
            'is_new_beneficiary': 1 if transaction['is_new_beneficiary'] else 0,
            **behavioral_features
        }])
        print_result("Transaction Amount", f"${transaction['amount']:.2f}")
        print_result("Hour of Day", f"{transaction['hour_of_day']}:00")
        print_result("Is New Beneficiary", transaction['is_new_beneficiary'])
        print_result("Recent Device Changes", behavioral_features['device_changes_last_24h'])

        # --- Stage 2: Parallel Model Inference ---
        print_subheader("Stage 2: Model Inference")
        scaled_features = self.scaler.transform(features_for_prediction[['amount', 'hour_of_day']])
        anomaly_score = self.anomaly_detector.decision_function(scaled_features)[0]
        fraud_probability = self.fraud_predictor.predict_proba(features_for_prediction)[0][1]
        graph_risk_boost = self._analyze_graph_network(transaction)

        print_result("Anomaly Score (Isolation Forest)", f"{anomaly_score:.4f} (Note: Negative is more anomalous)")
        print_result("Fraud Probability (XGBoost)", f"{fraud_probability:.4f} (Note: Higher indicates fraud)")

        # --- Stage 3: Risk Synthesis ---
        print_subheader("Stage 3: Risk Synthesis")
        xgb_score_component = fraud_probability * 80
        anomaly_score_component = max(0, -anomaly_score) * 20
        final_risk_score = min(max(int(xgb_score_component + anomaly_score_component + graph_risk_boost), 0), 100)

        print_result("XGBoost Contribution", f"{xgb_score_component:.2f} / 80.00")
        print_result("Anomaly Contribution", f"{anomaly_score_component:.2f} / 20.00")
        print_result("Graph Network Penalty", f"{graph_risk_boost:.2f}")
        print_result("FINAL RISK SCORE (0-100)", final_risk_score, indent=0)

        # --- Stage 4: Final Decision ---
        print_subheader("Stage 4: Final Decision")
        if final_risk_score > 75: action = "BLOCK & ALERT"
        elif final_risk_score > 40: action = "STEP-UP AUTHENTICATION"
        else: action = "ALLOW"

        print_result("Recommended Action", action, indent=0)
        print("="*70)
        return {"risk_score": final_risk_score, "action": action}

# ==============================================================================
# 4. MAIN EXECUTION BLOCK
# ==============================================================================

if __name__ == "__main__":
    # Initialize the engine once
    engine = FraudDetectionEngine()

    # --- Define Sample Transactions to Simulate ---
    normal_transaction = {
        "transaction_id": "TXN1001", "sender_account": "account_A", "receiver_account": "account_B",
        "amount": 500, "hour_of_day": 14, "is_new_beneficiary": False
    }

    suspicious_transaction = {
        "transaction_id": "TXN2002", "sender_account": "account_A", "receiver_account": "account_F",
        "amount": 9500, "hour_of_day": 3, "is_new_beneficiary": True
    }

    high_risk_mule_transaction = {
        "transaction_id": "TXN3003", "sender_account": "account_G", "receiver_account": "account_E",
        "amount": 2500, "hour_of_day": 11, "is_new_beneficiary": True
    }

    # --- Run Predictions ---
    engine.predict(normal_transaction)
    engine.predict(suspicious_transaction)
    engine.predict(high_risk_mule_transaction)


    INITIALIZING PROJECT SENTINEL ENGINE
Status                      : Anomaly Detection model trained.
Status                      : Fraud Prediction (XGBoost) model trained.

Engine is ready for real-time analysis.

    ANALYSIS REPORT FOR TRANSACTION ID: TXN1001

--- Stage 1: Feature Engineering ---
  Transaction Amount          : $500.00
  Hour of Day                 : 14:00
  Is New Beneficiary          : False
  Recent Device Changes       : 0

--- Stage 2: Model Inference ---
  Graph Analysis Alert        : No links to known fraud networks found.
  Anomaly Score (Isolation Forest): 0.1176 (Note: Negative is more anomalous)
  Fraud Probability (XGBoost) : 0.0009 (Note: Higher indicates fraud)

--- Stage 3: Risk Synthesis ---
  XGBoost Contribution        : 0.07 / 80.00
  Anomaly Contribution        : 0.00 / 20.00
  Graph Network Penalty       : 0.00
FINAL RISK SCORE (0-100)    : 0

--- Stage 4: Final Decision ---
Recommended Action          : ALLOW

    ANALYSIS REPORT FOR TRANS